In [1]:
!pip install lime

     |████████████████████████████████| 275 kB 1.8 MB/s eta 0:00:01
     |████████████████████████████████| 13.4 MB 1.2 MB/s eta 0:00:01    |██████▍                         | 2.7 MB 4.4 MB/s eta 0:00:03
     |████████████████████████████████| 317 kB 2.8 MB/s eta 0:00:01
     |████████████████████████████████| 227 kB 2.6 MB/s eta 0:00:01
Using legacy 'setup.py install' for lime, since package 'wheel' is not installed.
    Running setup.py install for lime ... done
You should consider upgrading via the '/Users/qika/Documents/Code/college/coolyeah/NLP/ScamDetection_NLP/venv/bin/python3 -m pip install --upgrade pip' command.


In [3]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load IndoBERT model and tokenizer
model_name = 'indobenchmark/indobert-base-p1'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

/Users/qika/Documents/Code/college/coolyeah/NLP/ScamDetection_NLP/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/qika/Documents/Code/college/coolyeah/NLP/ScamDetection_NLP/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from lime.lime_text import LimeTextExplainer
import numpy as np

class FraudReasoningXAI:
    def __init__(self, classifier_pipeline, tokenizer, class_names=['not scam', 'neutral', 'scam']):
        """
        classifier_pipeline: Fungsi atau Model Stage 2 yang menerima list teks dan output probabilitas
        tokenizer: Tokenizer dari IndoBERT
        """
        self.classifier_pipeline = classifier_pipeline
        self.tokenizer = tokenizer
        self.class_names = class_names
        self.explainer = LimeTextExplainer(class_names=class_names)

    def _predict_proba_wrapper(self, texts):
        """
        Wrapper untuk memastikan format output sesuai keinginan LIME.
        LIME butuh fungsi yang menerima list string dan return numpy array of probabilities.
        """
        # --- PENTING: ---
        # Di sini nanti kamu panggil Model Stage 2 (Classification) kamu.
        # Karena model stage 2 belum ada kodenya, saya buat Dummy Logic 
        # agar kode ini bisa jalan (Run-able) untuk demonstrasi.
        
        results = []
        for text in texts:
            # LOGIKA DUMMY (Hapus ini jika Model Stage 2 sudah masuk)
            # Simulasi: Jika ada kata 'hadiah', 'bayar', 'isi', 'pulsa' -> Scam chance tinggi
            scam_keywords = ['hadiah', 'bayar', 'transfer', 'pulsa', 'link', 'grup', 'biaya']
            count = sum(1 for word in scam_keywords if word in text.lower())
            
            if count >= 2:
                # High chance scam
                results.append([0.05, 0.05, 0.9]) # [not scam, neutral, scam]
            elif count == 1:
                # Maybe scam
                results.append([0.1, 0.4, 0.5])
            else:
                # Not scam
                results.append([0.8, 0.15, 0.05])
                
        return np.array(results)

    def explain_text(self, text_input, num_features=5):
        """
        Fungsi utama untuk menghasilkan reasoning.
        """
        # Generate explanation menggunakan LIME
        # Kita pass fungsi predict_proba kita ke LIME
        exp = self.explainer.explain_instance(
            text_input, 
            self._predict_proba_wrapper, # Ganti ini dengan model.predict_proba kamu nanti
            num_features=num_features,
            top_labels=1
        )
        
        # Ambil label prediksi teratas (misal: Scam)
        top_label_idx = exp.top_labels[0]
        predicted_label = self.class_names[top_label_idx]
        
        # Ambil kata-kata yang berkontribusi positif terhadap label tersebut
        explanation_list = exp.as_list(label=top_label_idx)
        
        # Filter hanya kata yang bobotnya positif (mendukung prediksi)
        keywords = [word for word, weight in explanation_list if weight > 0]
        
        return self._generate_natural_language_reasoning(predicted_label, keywords, text_input)

    def _generate_natural_language_reasoning(self, label, keywords, original_text):
        """
        Mengubah hasil matematis LIME menjadi kalimat penjelasan (Reasoning).
        """
        if not keywords:
            return f"Sistem mengklasifikasikan pesan ini sebagai **{label.upper()}**, namun pola spesifik sulit ditentukan secara individu."

        keywords_str = ", ".join([f"'{k}'" for k in keywords])
        
        reasoning = ""
        if label == 'scam':
            reasoning = (
                f"Pesan terdeteksi sebagai **SCAM**. \n"
                f"**Analisis:** Sistem menemukan indikasi penipuan kuat berdasarkan penggunaan kata kunci: **{keywords_str}**. \n"
                f"Kata-kata ini sering muncul dalam pola penipuan (seperti permintaan uang, urgensi, atau hadiah palsu) pada dataset historis."
            )
        elif label == 'neutral':
            reasoning = (
                f"Pesan dikategorikan **NETRAL**. \n"
                f"Meskipun mengandung kata **{keywords_str}**, konteksnya tidak cukup kuat untuk dianggap sebagai penipuan atau pesan resmi yang valid."
            )
        else: # Not Scam / Info Resmi
            reasoning = (
                f"Pesan ini diklasifikasikan **AMAN (Not Scam)**. \n"
                f"Struktur kalimat dan kata kunci seperti **{keywords_str}** konsisten dengan format informasi atau komunikasi wajar."
            )
            
        return reasoning

# --- CONTOH PENGGUNAAN ---

# 1. Inisialisasi Explainer (Menggunakan tokenizer IndoBERT yang sudah kamu load sebelumnya)
xai_module = FraudReasoningXAI(
    classifier_pipeline=None, # Nanti diisi model klasifikasi kamu
    tokenizer=tokenizer 
)

# 2. Teks Query (Contoh kasus)
query_text = "Halo Kak, kami dari admin grup 'Info Crypto Valid'. Anda harus bayar biaya langganan bulanan 200rb untuk tetap dapat info coin yang bakal pump."

# 3. Jalankan Reasoning
reasoning_output = xai_module.explain_text(query_text)

print("="*50)
print("HASIL TEST xAI REASONING")
print("="*50)
print(f"Input Text: {query_text}\n")
print(reasoning_output)

HASIL TEST xAI REASONING
Input Text: Halo Kak, kami dari admin grup 'Info Crypto Valid'. Anda harus bayar biaya langganan bulanan 200rb untuk tetap dapat info coin yang bakal pump.

Pesan terdeteksi sebagai **SCAM**. 
**Analisis:** Sistem menemukan indikasi penipuan kuat berdasarkan penggunaan kata kunci: **'biaya', 'grup', 'bayar', 'tetap', 'Halo'**. 
Kata-kata ini sering muncul dalam pola penipuan (seperti permintaan uang, urgensi, atau hadiah palsu) pada dataset historis.
